# Stage 3 — 门控与 H2（Figure 2）

> 对应 §8.3.5、§10.3、§13.4。

29 个系数可以直接印进论文：「什么让药物证据对天然产物可信」变成一张可读的表。
**这张图只有在门控是 29 参数 logistic 时才画得出来。**


In [ ]:
# 让 notebook 能 import sparc（无需 pip install -e .）
import sys, json
from pathlib import Path
CODE_ROOT = Path.cwd().parent if Path.cwd().name == 'experiments' else Path.cwd()
sys.path.insert(0, str(CODE_ROOT))

from sparc.common import load_experiment_config
cfg = load_experiment_config(stage='notebook', run_name='interactive')
print('冻结配置指纹：'); print(json.dumps(cfg.freeze_manifest(), indent=1))


## 1. 28 维特征清单


In [ ]:
manifest = cfg.gate_manifest
for i, (name, group, pool, sign) in enumerate(zip(manifest.names, manifest.groups, manifest.pools, manifest.sign_priors), 1):
    print(f'{i:2d}  {name:28s} {group:12s} {pool:6s} 先验符号 {sign:+d}')


## 2. 训练门控

```bash
python scripts/run_s3_gate.py --run-name s3_v1 --retrieval-ckpt s2_v1 --forced-retrieval-diagnostic
```


In [ ]:
gate_path = cfg.paths.stage_outputs('s3_gate') / 's3_v1_gate.json'
if gate_path.is_file():
    result = json.loads(gate_path.read_text(encoding='utf-8'))
    print('H2 通过 =', result['h2']['passed'])
    print('符号翻转 =', result['gate_coefficients']['n_sign_flips'], '/ 28（上限 2）')
    print('翻转的特征：', result['gate_coefficients']['sign_flips'])
    if 'c_forced_retrieval' in result:
        print('C_FORCED_RETRIEVAL:', result['c_forced_retrieval'])
else:
    print('尚未跑 Stage 3。')


## 3. Figure 2 — 门控系数图


In [ ]:
from sparc.eval.figures import plot_gate_coefficients

if gate_path.is_file():
    plot_gate_coefficients(result['gate_coefficients'],
                           cfg.paths.get('figures') / 'figure2_gate_coefficients.png')


> **符号翻转的正确读法**（§16 R5）：翻转不等于方法失败。
> 符号翻转常常是**未识别泄漏通道**的指示器 —— 先排查泄漏，再判定 H2。
